Using the LeNet5 model on the MNIST data set to learn the different perfomance metrics that can be used to test a DL model

In [1]:
# import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torchvision
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# See if GPU is available and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# load the MNIST dataset
# Convert images to tensors
transform = transforms.ToTensor()

# Training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Test dataset
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [ ]:
# the LeNet-5 model

class LeNet5(nn.Module):

    def __init__(self):
        super().__init__()

        # Convolution layer 1
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=6,
            kernel_size=5,
            padding=2
        )

        # Pooling layer
        self.pool = nn.AvgPool2d(
            kernel_size=2,
            stride=2
        )

        # Convolution layer 2
        self.conv2 = nn.Conv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5
        )

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        # Activation function
        self.relu = nn.ReLU()


    def forward(self, x):

        # First convolution
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        # Second convolution
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        # Flatten the image data
        x = torch.flatten(x, 1)

        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        # Final output
        x = self.fc3(x)

        return x

# Create the model
model = LeNet5().to(device)

# print the contents of the model
print(model)

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
  (relu): ReLU()
)


In [6]:
# Loss function and optimizer

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [7]:
# Training the model
epochs = 5

for epoch in range(epochs):

    # Put model into training mode
    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Add loss
        running_loss += loss.item()


    average_loss = running_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{epochs}, "
        f"Loss: {average_loss:.4f}"
    )


print("Training complete!")

Epoch 1/5, Loss: 0.3562
Epoch 2/5, Loss: 0.1056
Epoch 3/5, Loss: 0.0724
Epoch 4/5, Loss: 0.0577
Epoch 5/5, Loss: 0.0487
Training complete!


In [ ]:
def evaluate_model(model, test_loader, device):
    """
    Evaluate a pytorch classification model
    
    calculates:
        - Accuracy
        - Precision
        - Recall
        - F1 Score
    
    and creates a confusion matrix
    """

    #-------------------------------------------------------
    # test the model
    #-------------------------------------------------------


    # put the model into evaluation mode
    model.eval()

    # Create list to store true and predicted labels
    true_labels = []
    predicted_labels = []

    # no gradient calculation needed during evaluation
    with torch.no_grad():

        for inputs, labels in test_loader:

            # Move data to CPU/GPU
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)

            # Get predicted class
            _, predicted = torch.max(outputs.data, 1)

            # Store predictions and true labels
            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predicted.cpu().numpy())


    #-------------------------------------------------------
    # Calculate metrics
    #-------------------------------------------------------

    # calculate accuracy
    accuracy = accuracy_score(true_labels, predicted_labels)

    # calculate precision
    precision = precision_score(true_labels, 
                                predicted_labels, 
                                average='weighted',
                                zero_division=0)

    # calculate recall
    recall = recall_score(true_labels,
                          predicted_labels,
                          average='weighted',
                          zero_division=0)

    # calculate F1 score
    f1 = f1_score(true_labels,
                  predicted_labels,
                  average='macro',
                  zero_division=0)

    # Create confusion matrix
    cm = confusion_matrix(true_labels, predicted_labels)

    #-------------------------------------------------------
    # Print results
    #-------------------------------------------------------

    print("==== Model Performance ====")

    print(f"Accuracy : {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall   : {recall * 100:.2f}%")
    print(f"F1 Score : {f1 * 100:.2f}%")

    print("\nConfusion Matrix:")
    print(cm)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": cm
    }

In [9]:
# test the model
results = evaluate_model(
    model,
    test_loader,
    device
)

==== Model Performance ====
Accuracy : 98.61%
Precision: 98.62%
Recall   : 98.61%
F1 Score : 98.58%

Confusion Matrix:
[[ 972    0    0    0    1    1    0    1    3    2]
 [   0 1131    0    0    0    0    1    1    2    0]
 [   3    1 1022    1    1    0    0    3    1    0]
 [   0    0    3  991    0    8    0    4    2    2]
 [   0    0    0    0  982    0    0    0    0    0]
 [   1    0    0    2    0  886    2    1    0    0]
 [   7    2    0    0    5   15  929    0    0    0]
 [   0    3    3    1    2    0    0 1013    1    5]
 [   2    0    1    6    3    9    0    3  941    9]
 [   0    0    0    1    9    3    0    2    0  994]]
